In [40]:
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

EXT_TRIG = {
    'shortest': 'longest', 'longest': 'shortest',
    'tallest': 'shortest',
    'lightest': 'heaviest', 'heaviest': 'lightest',
    'smallest': 'largest', 'largest': 'smallest',
    'biggest': 'smallest',
    'earliest': 'latest', 'latest': 'earliest',
    'oldest': 'youngest', 'youngest': 'oldest',
    'worst': 'best', 'best': 'worst',
    'lowest': 'highest', 'highest': 'lowest',
    'most': 'least', 'least': 'most',
    'higher': 'lower', 'lower': 'higher',
    'more': 'less', 'less': 'more',
    'larger': 'smaller', 'smaller': 'larger',
    'older': 'younger', 'younger': 'older',
    'bigger': 'smaller',
    'equal': 'different', 'different': 'equal',
    'same': 'different',
    'first': 'last', 'last': 'first',
    'second': 'second to last',
    'third': 'third to last',
    'only': 'not only',
    'never': 'always', 'always': 'never',
    'all': 'none', 'none': 'all',
    'each': 'none', 'every': 'none',
    'was': 'was not', 'was not': 'was',
    'is': 'is not', 'is not': 'is',
    'are': 'are not', 'are not': 'are',
    'were': 'were not', 'were not': 'were',
    'sum': 'average', 'average': 'sum',
    'total': 'average',
    'previous': 'next', 'next': 'previous',
    'before': 'after', 'after': 'before'
}

def load_tagged(path: str):
    df = pd.read_csv(path, sep='\t')
    ner_idx = {
        str(r.utterance).lower().strip(): list(zip(str(r.tokens).split('|'), str(r.nerTags).split('|')))
        for r in df.itertuples()
    }
    return df, ner_idx

def make_S_minus(utterance: str) -> str:
    s = utterance.lower()
    for k, v in EXT_TRIG.items():
        if k in s:
            return s.replace(k, v, 1)
    return s

def find_evidence(tbl: pd.DataFrame, utterance: str, target: str = '') -> tuple[list, list]:
    # 1. Формируем словарь токенов (вопрос + ответы)
    tokens = set(utterance.lower().split())
    for t in target.split('|'):
        tokens.update(norm(t).split())

    linked_rows, linked_cols = set(), set()

    # --- ДОБАВЛЕНО: Проверка заголовков столбцов ---
    for col in tbl.columns:
        if set(str(col).lower().split()) & tokens:
            linked_cols.add(col)
    # ----------------------------------------------

    # 2. Поиск в ячейках (без изменений)
    for r_idx, row in tbl.iterrows():
        for c_idx, val in enumerate(row):
            cell_words = set(str(val).lower().split())
            if cell_words & tokens:
                linked_rows.add(r_idx)
                linked_cols.add(tbl.columns[c_idx])

    # 3. Очистка (Pruning)
    important_rows = [r for r in linked_rows
                      if any(set(str(tbl.at[r, c]).lower().split()) & tokens
                             for c in linked_cols)]

    return important_rows, list(linked_cols)


def make_T_plus(tbl: pd.DataFrame, utterance: str) -> pd.DataFrame:
    """Оставляет только улики (important_rows × important_cols), перемешивает."""
    important_rows, important_cols = find_evidence(tbl, utterance)
    
    cols = important_cols if important_cols else list(tbl.columns)
    rows = important_rows if important_rows else list(tbl.index)
    
    t_plus = tbl.loc[rows, cols].sample(frac=1, axis=0).sample(frac=1, axis=1)
    return t_plus.reset_index(drop=True)


# ─── T-: Саботаж улики ────────────────────────────────────────────────────────
def make_T_minus(tbl: pd.DataFrame, utterance: str, all_contexts: list) -> pd.DataFrame:
    import random
    important_rows, important_cols = find_evidence(tbl, utterance)
    
    if not important_rows and not important_cols:
        fallback_ctx = random.choice(all_contexts)
        try:
            return pd.read_csv(os.path.join('data', fallback_ctx), quotechar='"', on_bad_lines='skip')
        except:
            return tbl
    
    t_minus = tbl.reset_index(drop=True).copy().astype(object)  # ← фикс здесь
    
    mask_row = bool(important_rows) and (not important_cols or random.random() > 0.5)
    if mask_row:
        r = random.choice(important_rows)
        t_minus.loc[r, :] = '[MASK]'
    else:
        c = random.choice(important_cols)
        t_minus[c] = '[MASK]'
    
    return t_minus


def test_drive(n=1):
    tagged_path = 'data/tagged/data/training.tagged'
    df, ner_index = load_tagged(tagged_path)
    all_contexts = df['context'].tolist()  # для fallback T-
    df = df.head(n)

    pd.set_option('display.max_columns', 5)
    pd.set_option('display.width', 120)
    pd.set_option('display.max_colwidth', 15)

    for i, row in df.iterrows():
        q, target, ctx = row['utterance'], str(row['targetValue']), row['context']
        try:
            s_minus = make_S_minus(q)
            tbl     = pd.read_csv(os.path.join('data', ctx))
            t_plus  = make_T_plus(tbl, q)
            t_minus = make_T_minus(tbl, q, all_contexts)

            print(f"\n{'='*30} Пример №{i+1} {'='*30}")
            print(f"S:  {q}\nS-: {s_minus}\nTarget: {target}")
            print(f"\n--- T (оригинал) ---\n{tbl.head(10).to_string(index=False)}")
            print(f"\n--- T+ ---\n{t_plus.to_string(index=False)}")
            print(f"\n--- T- ---\n{t_minus.head(10).to_string(index=False)}")

        except Exception as e:
            print(f"\n--- Пример №{i+1} --- ОШИБКА: {e}")

if __name__ == "__main__":
    test_drive(1)


============================== Пример №1 ==============================
S:  what was the last year where this team was a part of the usl a-league?
S-: what was the first year where this team was a part of the usl a-league?
Target: 2004

--- T (оригинал) ---
 Year  Division              League Regular Season        Playoffs        Open Cup Avg. Attendance
 2001         2        USL A-League   4th, Western   Quarterfinals Did not qualify           7,169
 2002         2        USL A-League   2nd, Pacific       1st Round Did not qualify           6,260
 2003         2        USL A-League   3rd, Pacific Did not qualify Did not qualify           5,871
 2004         2        USL A-League   1st, Western   Quarterfinals       4th Round           5,628
 2005         2  USL First Division            5th   Quarterfinals       4th Round           6,028
 2006         2  USL First Division           11th Did not qualify       3rd Round           5,575
 2007         2  USL First Division            2

In [25]:
tagged_path = 'data/tagged/data/training.tagged'
df, ner_index = load_tagged(tagged_path) 

# 1. Создаем подмножество данных
batch_500 = df.head(500).copy()

# 2. Добавляем новую колонку ИМЕННО в batch_500
batch_500['masked_s'] = batch_500['utterance'].apply(lambda x: mask_entities(x, ner_index))

# 3. Список нужных колонок
columns_to_keep = ['utterance', 'masked_s', 'targetValue', 'context']

# 4. Фильтруем колонки в batch_500 (используем только те, что создали)
existing_columns = [col for col in columns_to_keep if col in batch_500.columns]

# ВАЖНО: берем данные из batch_500, а не из df
final_batch = batch_500[existing_columns].reset_index().rename(columns={'index': 'id'})
existing_columns = [col for col in columns_to_keep if col in batch_500.columns]
final_batch = batch_500[existing_columns].reset_index().rename(columns={'index': 'id'})

# КРИТИЧЕСКИЙ ШАГ: Сохранение на диск
# index=False нужен, чтобы пандас не создавал еще один лишний столбец с номерами строк
final_batch.to_excel('batch_for_gen.xlsx', index=False)

print("Файл 'batch_for_gen.xlsx' успешно создан в папке с проектом!")
print(f"Файл готов! Обработано колонок: {len(existing_columns)}")
print(final_batch.head())

Файл 'batch_for_gen.xlsx' успешно создан в папке с проектом!
Файл готов! Обработано колонок: 4
   id       utterance        masked_s     targetValue         context
0   0  what was th...  what was [M...            2004  csv/204-csv...
1   1  in what cit...  in what cit...  Bangkok, Th...  csv/204-csv...
2   2  which team ...  which team ...     Wolfe Tones  csv/204-csv...
3   3  how many mo...  how many mo...          12,467  csv/203-csv...
4   4  who was the...  who was the...    Derby County  csv/204-csv...


In [13]:
import pandas as pd
import os

# 1. Загрузка данных
# Убедитесь, что функции load_tagged и mask_entities определены выше в коде
df, ner_index = load_tagged('data/tagged/data/training.tagged')

# 2. ГЕНЕРАЦИЯ ОТСУТСТВУЮЩЕЙ КОЛОНКИ
# Это обязательный шаг, если в исходном .tagged файле нет готовых масок
print("Генерация masked_s... Это может занять некоторое время.")
df['masked_s'] = df['utterance'].apply(lambda x: mask_entities(x, ner_index))

# 3. Подготовка папки
output_dir = 'batchs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 4. Параметры цикла
batch_size = 500
total_rows = len(df)
# Список колонок, которые вы хотели видеть в батчах
columns_to_keep = ['utterance', 'masked_s', 'targetValue']

# 5. Цикл нарезки
for start_idx in range(0, total_rows, batch_size):
    end_idx = start_idx + batch_size
    
    # Берем срез
    batch = df.iloc[start_idx:end_idx].copy()
    
    # Оставляем только нужные колонки (теперь masked_s точно есть)
    existing_cols = [col for col in columns_to_keep if col in batch.columns]
    batch_to_save = batch[existing_cols].reset_index().rename(columns={'index': 'id'})
    
    # Сохранение
    filename = f"batch_{start_idx}_{min(end_idx - 1, total_rows - 1)}.xlsx"
    file_path = os.path.join(output_dir, filename)
    batch_to_save.to_excel(file_path, index=False)
    
    print(f"Сохранен: {filename}")

print("\nГотово! Все файлы в папке 'batchs' теперь содержат колонку masked_s.")

Генерация masked_s... Это может занять некоторое время.
Сохранен: batch_0_499.xlsx
Сохранен: batch_500_999.xlsx
Сохранен: batch_1000_1499.xlsx
Сохранен: batch_1500_1999.xlsx
Сохранен: batch_2000_2499.xlsx
Сохранен: batch_2500_2999.xlsx
Сохранен: batch_3000_3499.xlsx
Сохранен: batch_3500_3999.xlsx
Сохранен: batch_4000_4499.xlsx
Сохранен: batch_4500_4999.xlsx
Сохранен: batch_5000_5499.xlsx
Сохранен: batch_5500_5999.xlsx
Сохранен: batch_6000_6499.xlsx
Сохранен: batch_6500_6999.xlsx
Сохранен: batch_7000_7499.xlsx
Сохранен: batch_7500_7999.xlsx
Сохранен: batch_8000_8499.xlsx
Сохранен: batch_8500_8999.xlsx
Сохранен: batch_9000_9499.xlsx
Сохранен: batch_9500_9999.xlsx
Сохранен: batch_10000_10499.xlsx
Сохранен: batch_10500_10999.xlsx
Сохранен: batch_11000_11499.xlsx
Сохранен: batch_11500_11999.xlsx
Сохранен: batch_12000_12499.xlsx
Сохранен: batch_12500_12999.xlsx
Сохранен: batch_13000_13499.xlsx
Сохранен: batch_13500_13999.xlsx
Сохранен: batch_14000_14151.xlsx

Готово! Все файлы в папке 'batch

In [31]:
df, ner_index = load_tagged('data/tagged/data/training.tagged')
df.id

0            nt-0
1            nt-1
2            nt-2
3            nt-3
4            nt-4
           ...   
14147    nt-14147
14148    nt-14148
14149    nt-14149
14150    nt-14150
14151    nt-14151
Name: id, Length: 14152, dtype: object

In [21]:
import pandas as pd

# 1. Выбираем нужные столбцы
# context — это путь к csv файлу таблицы
export_df = df[['id', 'context']].copy()

# 2. Сохраняем в Excel
output_path = 'training_metadata.xlsx'
export_df.to_excel(output_path, index=False)

print(f"Файл успешно сохранен: {output_path}")

Файл успешно сохранен: training_metadata.xlsx


In [65]:
import pandas as pd
import os

# ─── Загрузка S+ датасета ─────────────────────────────────────────────────────
s_plus_df = pd.read_excel('batchs/final_merged_file.xlsx')

# ─── Загрузка tagged для S- ───────────────────────────────────────────────────
df, ner_index = load_tagged('data/tagged/data/training.tagged')
all_contexts  = df['context'].tolist()

# ─── Сборка финального датасета ───────────────────────────────────────────────
records = []

for _, row in s_plus_df.iterrows():
    q      = row['utterance']
    s_plus = row['S_p']
    target = str(row['targetValue'])
    ctx    = row['context']

    try:
        tbl = pd.read_csv(os.path.join('data', ctx), quotechar='"', on_bad_lines='skip', engine='python')
        t_plus  = make_T_plus(tbl, q)
        t_minus = make_T_minus(tbl, q, all_contexts)
        s_minus = make_S_minus(q)

        records.append({
            "id":      row['id'],
            "S":       q,
            "S_plus":  s_plus,
            "S_minus": s_minus,
            "T":       tbl.to_csv(index=False),       # ⟨S, T⟩
            "T_plus":  t_plus.to_csv(index=False),    # ⟨S+, T+⟩
            "T_minus": t_minus.to_csv(index=False),   # ⟨S, T-⟩
            "answer":  target
        })

    except Exception as e:
        print(f"Пропуск id={row['id']}: {e}")

# ─── Сохранение ───────────────────────────────────────────────────────────────
out = pd.DataFrame(records)
out.to_csv('data/contrastive_dataset.tsv', sep='\t', index=False)
print(f"✅ Сохранено {len(records)} примеров → data/contrastive_dataset.tsv")

✅ Сохранено 14152 примеров → data/contrastive_dataset.tsv


In [67]:
df_check = pd.read_csv('data/contrastive_dataset.tsv', sep='\t')

print(f"Строк: {len(df_check)}")
print(f"Столбцы: {list(df_check.columns)}")
print(f"Пропуски:\n{df_check.isnull().sum()}")
print(f"\nПервая запись:")
print(df_check.iloc[0][['S', 'S_plus', 'S_minus', 'answer','T']])

Строк: 14152
Столбцы: ['id', 'S', 'S_plus', 'S_minus', 'T', 'T_plus', 'T_minus', 'answer']
Пропуски:
id         0
S          0
S_plus     0
S_minus    0
T          0
T_plus     0
T_minus    0
answer     1
dtype: int64

Первая запись:
S          what was th...
S_plus     In which [M...
S_minus    what was th...
answer               2004
T          Year,Divisi...
Name: 0, dtype: object


In [59]:
import json

# CSV без таблиц
df_check[['id', 'S', 'S_plus', 'S_minus', 'answer']].to_csv(
    'data/contrastive_dataset.csv', index=False, encoding='utf-8-sig'
)
# JSON только с таблицами
tables = df_check[['id', 'T', 'T_plus', 'T_minus']].to_dict(orient='records')
with open('data/contrastive_tables.json', 'w', encoding='utf-8') as f:
    json.dump(tables, f, ensure_ascii=False, indent=2)

print(f"✅ CSV: {len(df_check)} строк → data/contrastive_dataset.csv")
print(f"✅ JSON: {len(tables)} записей → data/contrastive_tables.json")

✅ CSV: 14152 строк → data/contrastive_dataset.csv
✅ JSON: 14152 записей → data/contrastive_tables.json


In [176]:
import pandas as pd

def count_mask_mismatches(file_path: str) -> int:
    df = pd.read_excel(file_path)
    # Считаем количество '[MASK]' в столбцах и сравниваем
    return (df['masked_s'].str.count(r'\[MASK\]') != df['S_p'].str.count(r'\[MASK\]'))

mismatches = count_mask_mismatches('batchs/merged_file_S_p_teg.xlsx')


In [177]:
mismatches[mismatches == True]

Series([], dtype: bool)

In [150]:
# Сохраняем S_p с тегами
tagged_df = pd.DataFrame(ok)
tagged_df.to_excel('batchs/merged_file_S_p_teg.xlsx', index=False)
print(f"✅ Сохранено → batchs/merged_file_S_p_teg.xlsx")


✅ Сохранено → batchs/merged_file_S_p_teg.xlsx


In [53]:
df = pd.read_excel('batchs/final_merged_file_fixed.xlsx')
# Фильтруем строки, где количество масок разное, и сохраняем в новый файл
df[df['masked_s'].str.count(r'\[MASK\]') != df['S_p'].str.count(r'\[MASK\]')].to_excel('need_fix_2.xlsx', index=False)

In [33]:
df_fix = pd.read_excel('batchs/fix_file_2.xlsx')

df_fix.head()

,id,masked_s,S_p
0,229,"who held the position longer , t.v sivaraopant...",Who held the position for a longer duration—T....
1,1741,who is listed as the last responsible mister -...,Who is identified as the final responsible par...
2,1787,how many times did super tetris win an award ?,On how many occasions was Super Tetris the rec...
3,1830,what is the number of saves for the [MASK],How many saves did the [MASK] record?
4,1921,which bainbridge class destroyer was launched ...,Which destroyer of the Bainbridge class was la...


In [124]:
import pandas as pd

final_df = pd.('batchs/final_merged_file_fixed.xlsx')
fix_df   = pd.read_excel('batchs/need_fix.xlsx')

# джойним по masked_s вместо id
fix_map = fix_df.set_index('masked_s')['S_p'].to_dict()

# заменяем S_p в final_df где masked_s совпадает
final_df['S_p'] = final_df.apply(
    lambda row: fix_map.get(row['masked_s'], row['S_p']), axis=1
)

# проверка
fixed_count = final_df['masked_s'].isin(fix_map.keys()).sum()
print(f"Обновлено: {fixed_count} строк")

final_df.to_excel('batchs/final_merged_file_fixed_2.xlsx', index=False)
print("✅ Сохранено → batchs/final_merged_file_fixed.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'batchs/need_fix.xlsx'

In [182]:
def make_S_plus_tagged(s_p: str, utterance: str, ner_index: dict) -> str | None:
    pairs = ner_index.get(utterance.lower().strip(), [])
    
    # собираем сущности по порядку
    entities, prev_entity = [], []
    for token, tag in pairs:
        if tag != 'O':
            prev_entity.append((token, tag))
        else:
            if prev_entity:
                entities.append((' '.join(t for t, _ in prev_entity), prev_entity[0][1]))
                prev_entity = []
    if prev_entity:
        entities.append((' '.join(t for t, _ in prev_entity), prev_entity[0][1]))

    # проверка совпадения масок
    if s_p.count('[MASK]') != len(entities):
        return None  # не совпадает → пропуск

    result = s_p
    for text, tag in entities:
        result = result.replace('[MASK]', f'[{tag}]', 1)

    return result


# ─── Тест ────────────────────────────────────────────────────────────────────
s_plus_df    = pd.read_excel('batchs/S_p.xlsx')
_, ner_index = load_tagged('data/tagged/data/training.tagged')

ok, failed = [], []

for _, row in tqdm(s_plus_df.iterrows(), total=len(s_plus_df)):
    result = make_S_plus_tagged(row['S_p'], row['utterance'], ner_index)
    if result:
        ok.append({**row.to_dict(), 'S_plus_tagged': result})
    else:
        failed.append(row.to_dict())

print(f"✅ Успешно:  {len(ok)}")
print(f"⚠️  Failed:  {len(failed)}")

print(f"\nПример:")
print(f"S:      {ok[0]['utterance']}")
print(f"S_p:    {ok[0]['S_p']}")
print(f"Tagged: {ok[0]['S_plus_tagged']}")

100%|█████████████████████████████████████████████████████████████████████████| 14152/14152 [00:00<00:00, 22694.75it/s]

✅ Успешно:  14152
⚠️  Failed:  0

Пример:
S:      what was the last year where this team was a part of the usl a-league?
S_p:    In which [MASK] year did this team last belong to the [MASK] A-League?
Tagged: In which [DATE] year did this team last belong to the [MISC] A-League?


In [184]:
pd.DataFrame(ok).to_excel('batchs/S_tagged.xlsx', index=False)
print("✅ Сохранено → batchs/S_tagged.xlsx")

✅ Сохранено → batchs/S_tagged.xlsx


In [144]:
# проверка дублирований
duplicates = []
for item in ok:
    tagged = item['S_plus_tagged']
    # ищем паттерн: </TAG> word где word уже есть внутри тега
    import re
    matches = re.findall(r'<(\w+)>(.*?)</\1>\s+(\w+)', tagged)
    for tag, content, next_word in matches:
        if next_word.lower() in content.lower():
            duplicates.append(item)
            break

print(f"Дублирований: {len(duplicates)}")
print(duplicates[0]['S_plus_tagged'] if duplicates else "Нет")

Дублирований: 348
In which <DATE>the last year</DATE> year did this team last belong to the <MISC>usl</MISC> A-League?


In [188]:
tagged_df = pd.read_excel('batchs/S_p.xlsx')

tagged_df['S_minus'] = tagged_df['utterance'].apply(make_S_minus)

failed_s_minus = tagged_df[
    tagged_df['S_minus'] == tagged_df['utterance'].str.lower()
]

failed_s_minus.to_excel('batchs/failed_s_minus.xlsx', index=False)
print(f"⚠️  Без S-: {len(failed_s_minus)} строк → batchs/failed_s_minus.xlsx")

⚠️  Без S-: 1738 строк → batchs/failed_s_minus.xlsx


In [192]:
tagged_df      = pd.read_excel('batchs/S_tagged.xlsx')
failed_df      = pd.read_excel('batchs/failed_s_minus.xlsx').set_index('id')

def get_s_minus(row):
    s_minus = make_S_minus(row['utterance'])
    if s_minus == row['utterance'].lower():  # триггер не сработал
        return failed_df.loc[row['id'], 'inverted_question'] if row['id'] in failed_df.index else s_minus
    return s_minus

tagged_df['S_minus'] = tagged_df.apply(get_s_minus, axis=1)

tagged_df.to_excel('batchs/S_p_m.xlsx', index=False)
print(f"✅ S_minus добавлен → {len(tagged_df)} строк")

✅ S_minus добавлен → 14152 строк


In [9]:
import pandas as pd

tagged_df    = pd.read_excel('batchs/S_p_m.xlsx')
_, ner_index = load_tagged('data/tagged/data/training.tagged')
all_contexts = tagged_df['context'].tolist()

total        = len(tagged_df)
found, not_found = 0, 0

for _, row in tqdm(tagged_df.iterrows(), total=total):
    try:
        tbl = pd.read_csv(os.path.join('data', row['context']), 
                          quotechar='"', on_bad_lines='skip', engine='python')
        important_rows, important_cols = find_evidence(tbl, row['utterance'])
        if important_rows or important_cols:
            found += 1
        else:
            not_found += 1
    except:
        not_found += 1

print(f"Всего таблиц:        {total}")
print(f"Найдены улики:       {found}  ({found/total*100:.1f}%)")
print(f"Ничего не найдено:   {not_found}  ({not_found/total*100:.1f}%)")

NameError: name 'tqdm' is not defined

In [17]:
from tqdm import tqdm
import pandas as pd
import os

tagged_df = pd.read_excel('batchs/S_p_m.xlsx')
total     = len(tagged_df)
norm      = lambda v: str(v).replace(',', '').replace('.0', '').strip().lower()

stats = {'both': 0, 'only_answer': 0, 'only_question': 0, 'nothing': 0}

for _, row in tqdm(tagged_df.iterrows(), total=total):
    try:
        tbl      = pd.read_csv(os.path.join('data', row['context']),
                               quotechar='"', on_bad_lines='skip', engine='python')
        cell_set = {norm(v) for v in tbl.values.flatten()}

        # ищем слова вопроса
        q_tokens    = {norm(t) for t in str(row['utterance']).split()}
        answer_hit  = any(norm(t) in cell_set for t in str(row['targetValue']).split('|'))
        question_hit = bool(q_tokens & cell_set)

        if answer_hit and question_hit:
            stats['both'] += 1
        elif answer_hit:
            stats['only_answer'] += 1
        elif question_hit:
            stats['only_question'] += 1
        else:
            stats['nothing'] += 1

    except:
        stats['nothing'] += 1

print(f"Всего таблиц:                        {total}")
print(f"Найдены и ответ и вопрос:            {stats['both']}  ({stats['both']/total*100:.1f}%)")
print(f"Найден только ответ:                 {stats['only_answer']}  ({stats['only_answer']/total*100:.1f}%)")
print(f"Найдены только слова вопроса:        {stats['only_question']}  ({stats['only_question']/total*100:.1f}%)")
print(f"Ничего не найдено:                   {stats['nothing']}  ({stats['nothing']/total*100:.1f}%)")

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:15<00:00, 903.47it/s]

Всего таблиц:                        14152
Найдены и ответ и вопрос:            291  (2.1%)
Найден только ответ:                 1969  (13.9%)
Найдены только слова вопроса:        402  (2.8%)
Ничего не найдено:                   11490  (81.2%)


In [19]:
found, not_found = 0, 0

norm = lambda v: str(v).replace(',', '').replace('.0', '').strip().lower()



for _, row in tqdm(tagged_df.iterrows(), total=total):

    try:

        tbl    = pd.read_csv(os.path.join('data', row['context']),

                             quotechar='"', on_bad_lines='skip', engine='python')

        target = norm(str(row['targetValue']))

        hits   = any(norm(v) == target for v in tbl.values.flatten())

        if hits:

            found += 1

        else:

            not_found += 1

    except:

        not_found += 1



print(f"Всего таблиц:               {total}")

print(f"targetValue найден:          {found}  ({found/total*100:.1f}%)")

print(f"targetValue не найден:       {not_found}  ({not_found/total*100:.1f}%)")

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:14<00:00, 948.36it/s]

Всего таблиц:               14152
targetValue найден:          2212  (15.6%)
targetValue не найден:       11940  (84.4%)


In [23]:
tagged_df    = pd.read_excel('batchs/S_p_m.xlsx')
_, ner_index = load_tagged('data/tagged/data/training.tagged')
all_contexts = tagged_df['context'].tolist()

total        = len(tagged_df)
found, not_found = 0, 0

for _, row in tqdm(tagged_df.iterrows(), total=total):
    try:
        tbl = pd.read_csv(os.path.join('data', row['context']), 
                          quotechar='"', on_bad_lines='skip', engine='python')
        important_rows, important_cols = find_evidence(tbl, row['utterance'])
        if important_rows or important_cols:
            found += 1
        else:
            not_found += 1
    except:
        not_found += 1

print(f"Всего таблиц:        {total}")
print(f"Найдены улики:       {found}  ({found/total*100:.1f}%)")
print(f"Ничего не найдено:   {not_found}  ({not_found/total*100:.1f}%)")

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:23<00:00, 608.60it/s]

Всего таблиц:        14152
Найдены улики:       3989  (28.2%)
Ничего не найдено:   10163  (71.8%)


In [42]:
found_both = []

for _, row in tqdm(tagged_df.iterrows(), total=total):
    try:
        tbl = pd.read_csv(os.path.join('data', row['context']),
                          quotechar='"', on_bad_lines='skip', engine='python')
        imp_rows, imp_cols = find_evidence(tbl, row['utterance'], str(row['targetValue']))

        if imp_rows or imp_cols:
            found_both.append(row.to_dict())
    except:
        pass

print(f"Найдено: {len(found_both)} строк ({len(found_both)/total*100:.1f}%)")

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:24<00:00, 576.51it/s]

Найдено: 6605 строк (46.7%)


In [44]:
from IPython.display import display

for item in found_both[:5]:
    print(f"\n{'='*50}\nВОПРОС: {item['utterance']} | ОТВЕТ: {item['targetValue']}")
    
    # 1. Загрузка
    tbl = pd.read_csv(os.path.join('data', item['context']), quotechar='"', on_bad_lines='skip', engine='python')
    
    # 2. Поиск улик
    rows, cols = find_evidence(tbl, item['utterance'], str(item['targetValue']))
    
    # 3. Вывод
    print("--- ОРИГИНАЛ ---")
    display(tbl)
    
    print(f"--- УЛИКИ (Строки: {rows}, Колонки: {cols}) ---")
    display(tbl.loc[rows, cols] if rows and cols else "Ничего не найдено")


ВОПРОС: what was the last year where this team was a part of the usl a-league? | ОТВЕТ: 2004
--- ОРИГИНАЛ ---


,Year,Division,...,Open Cup,Avg. Attendance
0,2001,2,...,Did not qua...,"7,169"
1,2002,2,...,Did not qua...,"6,260"
2,2003,2,...,Did not qua...,"5,871"
3,2004,2,...,4th Round,"5,628"
4,2005,2,...,4th Round,"6,028"
5,2006,2,...,3rd Round,"5,575"
6,2007,2,...,2nd Round,"6,851"
7,2008,2,...,1st Round,"8,567"
8,2009,2,...,3rd Round,"9,734"
9,2010,2,...,3rd Round,"10,727"


--- УЛИКИ (Строки: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], Колонки: ['Regular Season', 'Year', 'League']) ---


,Regular Season,Year,League
0,"4th, Western",2001,USL A-League
1,"2nd, Pacific",2002,USL A-League
2,"3rd, Pacific",2003,USL A-League
3,"1st, Western",2004,USL A-League
4,5th,2005,USL First D...
5,11th,2006,USL First D...
6,2nd,2007,USL First D...
7,11th,2008,USL First D...
8,1st,2009,USL First D...
9,"3rd, USL (3rd)",2010,USSF D-2 Pr...



ВОПРОС: in what city did piotr's last 1st place finish occur? | ОТВЕТ: Bangkok, Thailand
--- ОРИГИНАЛ ---


...

,City,Passengers
0,United Stat...,"14,749"
3,"Canada, Sas...","2,282"



ВОПРОС: who was the opponent in the first game of the season? | ОТВЕТ: Derby County
--- ОРИГИНАЛ ---


,Date,Opponent,...,Attendance,Scorers
0,15 August 1987,Derby County,...,"17,204",—
1,18 August 1987,Coventry City,...,"09,380",—
2,22 August 1987,West Ham Un...,...,"08,073",Harford (2)
3,29 August 1987,Chelsea,...,"16,075",—
4,31 August 1987,Arsenal,...,"08,745",Wilson (pen)
5,5 September...,Oxford United,...,"06,804","Breacker, H..."
6,12 Septembe...,Everton,...,"08,124","Hill, B. Stein"
7,19 Septembe...,Charlton At...,...,"05,002",—
8,26 Septembe...,Queens Park...,...,"11,175",—
9,3 October 1987,Manchester ...,...,"09,137",Harford


--- УЛИКИ (Строки: [0, 23], Колонки: ['Opponent']) ---


,Opponent
0,Derby County
23,Derby County


In [48]:
# сохраняем найденные
found_df = pd.DataFrame(found_both)
found_df.to_excel('batchs/found_evidence.xlsx', index=False)

# строки для модели — которых НЕТ в found_both
found_ids  = set(found_df['id'])
model_df   = tagged_df[~tagged_df['id'].isin(found_ids)].reset_index(drop=True)
print(f"Для алгоритма: {len(found_df)} строк")
print(f"Для модели:    {len(model_df)} строк")

Для алгоритма: 6605 строк
Для модели:    7547 строк


In [54]:
import pandas as pd
import os
from tqdm import tqdm

def make_T_plus(tbl: pd.DataFrame) -> pd.DataFrame:
    return tbl.sample(frac=1, axis=0).sample(frac=1, axis=1).reset_index(drop=True)

tagged_df = pd.read_excel('batchs/S_p_m.xlsx')
os.makedirs('data/T_p', exist_ok=True)

failed = []

for _, row in tqdm(tagged_df.iterrows(), total=len(tagged_df)):
    try:
        tbl    = pd.read_csv(os.path.join('data', row['context']),
                             quotechar='"', on_bad_lines='skip', engine='python')
        t_plus = make_T_plus(tbl)
        t_plus.to_csv(f"data/T_p/{row['id']}_positive.csv", index=False, encoding='utf-8')
    except Exception as e:
        failed.append({'id': row['id'], 'error': str(e)})

print(f"✅ Успешно: {len(tagged_df) - len(failed)}")
print(f"⚠️  Failed: {len(failed)}")

if failed:
    pd.DataFrame(failed).to_excel('batchs/t_plus_failed.xlsx', index=False)

100%|███████████████████████████████████████████████████████████████████████████| 14152/14152 [00:36<00:00, 383.77it/s]

✅ Успешно: 14152
⚠️  Failed: 0
